## About Dataset
Context
This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content
5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset.
Columns

- asin - ID of the product, like B000FA64PK
- helpful - helpfulness rating of the review - example: 2/3.
- overall - rating of the product.
- reviewText - text of the review (heading).
- reviewTime - time of the review (raw).
- reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
- reviewerName - name of the reviewer.
- summary - summary of the review (description).
- unixReviewTime - unix timestamp.

Acknowledgements
This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

Inspiration
- Sentiment analysis on reviews.
- Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
- Fake reviews/ outliers.
- Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
- Any other interesting analysis

In [26]:
import numpy as np
import pandas as pd

In [27]:
df = pd.read_csv("/content/all_kindle_review.csv")

In [28]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [29]:
df = df[["reviewText","rating"]]

In [30]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [31]:
df.shape

(12000, 2)

In [32]:
# Check Missing values
df.isnull().sum()

,0
reviewText,0
rating,0


In [33]:
df["rating"].value_counts()

,count
rating,
5,3000
4,3000
3,2000
2,2000
1,2000


In [34]:
df["rating"]= df["rating"].apply(lambda x: 0 if x <3 else 1)

In [35]:
df["rating"].value_counts()

,count
rating,
1,8000
0,4000


In [36]:
# Lower all the cases
df["reviewText"] = df["reviewText"].apply(lambda x: x.lower())

In [39]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [48]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [41]:
from bs4 import BeautifulSoup

In [42]:
## Removing special characters
df["reviewText"] =df["reviewText"].apply(lambda x: re.sub('[^a-z A-Z 0-9-]+','',x))
## Remove the stopswords
df["reviewText"] =df["reviewText"].apply(lambda x: " ".join([y for y in x.split() if y not in stopwords.words("english")]))
## Remove url
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
## Remove html tags
df['reviewText']=df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())
## Remove any additional spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

In [43]:
df.head()


,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [44]:
## Lemmatizer
from nltk.stem import WordNetLemmatizer

In [45]:
lemmatizer = WordNetLemmatizer()

In [46]:
def lemmatize_words(text):
  return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [49]:
df["reviewText"] = df["reviewText"].apply(lambda x: lemmatize_words(x))

In [50]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [57]:
X = df.iloc[:,0]
y = df.iloc[:,-1]

In [58]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20)

In [59]:
X_train.shape

(9600,)

In [60]:
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train).toarray()
X_test_tfidf=tfidf.transform(X_test).toarray()

In [62]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [63]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)

In [64]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

In [65]:
y_pred_bow = nb_model_bow.predict(X_test_bow)
y_pred_tfidf = nb_model_tfidf.predict(X_test_tfidf)

In [66]:
confusion_matrix(y_test,y_pred_bow)

array([[509, 287],
       [737, 867]])

In [67]:
print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))

BOW accuracy:  0.5733333333333334


In [68]:
confusion_matrix(y_test,y_pred_tfidf)

array([[497, 299],
       [722, 882]])

In [70]:
print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))

TFIDF accuracy:  0.5745833333333333


## Word2Vec

In [73]:
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 21.4 MB/s eta 0:00:00


In [77]:
import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, KeyedVectors
wv=api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [78]:
def preprocess(text):
    return text.lower().split()

X_train_tokens = X_train.apply(preprocess)
X_test_tokens = X_test.apply(preprocess)

In [83]:
len(X_train_tokens)

9600

In [85]:
w2v_model = Word2Vec(sentences=X_train_tokens, vector_size=300, min_count=2)

In [92]:
print(len(w2v_model.wv.index_to_key))

15792


In [86]:
import numpy as np

def avg_word2vec(doc):
    vectors = [
        w2v_model.wv[word]
        for word in doc
        if word in w2v_model.wv.index_to_key
    ]

    if len(vectors) == 0:
        return np.zeros(w2v_model.vector_size)

    return np.mean(vectors, axis=0)

In [87]:
X_train_vec = np.vstack(X_train_tokens.apply(avg_word2vec))
X_test_vec  = np.vstack(X_test_tokens.apply(avg_word2vec))

In [89]:
X_train_vec.shape

(9600, 300)

In [93]:
from sklearn.naive_bayes import GaussianNB
nb_model_w2v=GaussianNB().fit(X_train_vec,y_train)


In [94]:
nb_model_w2v_pred = nb_model_w2v.predict(X_test_vec)

In [95]:
print("W2V accuracy: ",accuracy_score(y_test,nb_model_w2v_pred))

W2V accuracy:  0.6783333333333333
